In [ ]:
import numpy as np
import pandas as pd
import os, warnings, re
warnings.filterwarnings('ignore')

import cv2
from skimage.feature import local_binary_pattern, hog
from skimage import exposure

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Dataset = https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri

In [ ]:
# CELL 1 - Install & Import

# Path utama
BASE_FEATURE = '/content/drive/MyDrive/riset-citra/feature'
BASE_RESULT  = '/content/drive/MyDrive/riset-citra/result'
DATASET_DIR  = '/content/drive/MyDrive/riset-citra/dataset'

CLASSES      = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
N_FOLDS      = 5
RANDOM_STATE = 42
IMG_SIZE     = (224, 224)

# Folder feature
DIR_LBP     = f'{BASE_FEATURE}/lbp'
DIR_HOG     = f'{BASE_FEATURE}/hog'
DIR_HOGLBP  = f'{BASE_FEATURE}/hog-lbp'
DIR_PCA     = f'{BASE_FEATURE}/pca'

PCA_VARS    = [0.90, 0.95, 0.99]
PCA_LABELS  = ['90', '95', '99']

print("Setup selesai.")

In [ ]:
# CELL 2 - Rebuild folds

from sklearn.model_selection import StratifiedKFold
from collections import Counter

all_paths, all_labels = [], []
SPLITS = ['Training', 'Testing']

for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(DATASET_DIR, split, cls)
        if not os.path.exists(cls_path):
            continue
        for fname in sorted(os.listdir(cls_path)):
            fpath = os.path.join(cls_path, fname)
            if os.path.isfile(fpath) and fname.lower().endswith('.jpg'):
                all_paths.append(fpath)
                all_labels.append(cls)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(all_paths, all_labels))

print(f"Total gambar : {len(all_paths)}")
for cls, cnt in sorted(Counter(all_labels).items()):
    print(f"  {cls}: {cnt}")
print(f"Folds siap   : {len(folds)}")

In [ ]:
# CELL 3 - load_gray, extract_hog, extract_lbp

def load_gray(path, size=IMG_SIZE):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Tidak ditemukan: {path}")
    img = cv2.resize(img, size)
    if len(img.shape) == 2:
        return img
    if img.shape[2] == 3:
        if np.array_equal(img[:,:,0], img[:,:,1]) and \
           np.array_equal(img[:,:,1], img[:,:,2]):
            return img[:,:,0]
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    raise ValueError(f"Format tidak dikenali: {img.shape}")


def extract_hog(img_gray):
    feat = hog(
        img_gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False
    )
    return feat.astype('float32')


def extract_lbp(img_gray):
    RADIUS   = 2
    N_POINTS = 8 * RADIUS
    N_BINS   = N_POINTS + 2
    lbp      = local_binary_pattern(img_gray, N_POINTS, RADIUS, method='uniform')
    hist, _  = np.histogram(lbp.ravel(),
                             bins=np.arange(0, N_BINS + 1),
                             range=(0, N_BINS))
    hist     = hist.astype('float32')
    hist    /= (hist.sum() + 1e-6)
    return hist

def extract_hoglbp(img_gray):
    return np.hstack([extract_hog(img_gray), extract_lbp(img_gray)])

In [ ]:
# CELL 4 Ekstraksi Fitur HOG dan HOG-LBP

EXTRACTORS = {
    'hog'    : extract_hog,
    'hog-lbp': extract_hoglbp,
}

for feat_name, extractor in EXTRACTORS.items():
    feat_dir = f'{BASE_FEATURE}/{feat_name}'
    print(f"\n{'='*50}")
    print(f"  Ekstraksi: {feat_name.upper()}")
    print(f"{'='*50}")

    for fold_idx, (train_idx, val_idx) in enumerate(folds, 1):
        fold_dir = f'{feat_dir}/fold_{fold_idx}'
        os.makedirs(fold_dir, exist_ok=True)

        for split_name, indices in [('train', train_idx), ('val', val_idx)]:
            # Skip jika sudah ada
            out_path = f'{fold_dir}/{split_name}.npz'
            if os.path.exists(out_path):
                print(f"  [SKIP] {out_path} sudah ada")
                continue

            X, y, paths = [], [], []
            for i in tqdm(indices, desc=f'Fold {fold_idx} {split_name}'):
                try:
                    img_gray = load_gray(all_paths[i])
                    feat     = extractor(img_gray)
                    X.append(feat)
                    y.append(all_labels[i])
                    paths.append(all_paths[i])
                except Exception as e:
                    print(f"  [ERROR] {all_paths[i]}: {e}")

            X     = np.array(X,     dtype='float32')
            y     = np.array(y,     dtype=str)
            paths = np.array(paths, dtype=str)

            np.savez_compressed(out_path, X=X, y=y, paths=paths)
            print(f"  Tersimpan: {out_path}  shape={X.shape}")

print("\nEkstraksi HOG & HOG-LBP selesai!")

In [ ]:
#  CELL Ekstraksi Fitur LBP
from skimage.feature import local_binary_pattern
import numpy as np, os
from tqdm import tqdm

FEATURE_DIR = '/content/drive/MyDrive/riset-citra/feature/lbp'
RADIUS      = 2
N_POINTS    = 8 * RADIUS     # 16
N_BINS      = N_POINTS + 2   # 18

def extract_lbp(img_gray):
    lbp  = local_binary_pattern(img_gray, N_POINTS, RADIUS, method='uniform')
    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, N_BINS + 1),
        range=(0, N_BINS)
    )
    hist = hist.astype('float32')
    hist /= (hist.sum() + 1e-6)
    return hist


for fold_idx, (train_idx, val_idx) in enumerate(folds, 1):
    fold_dir = os.path.join(FEATURE_DIR, f'fold_{fold_idx}')
    os.makedirs(fold_dir, exist_ok=True)

    for split_name, indices in [('train', train_idx), ('val', val_idx)]:
        X, y, paths = [], [], []

        for i in tqdm(indices, desc=f'Fold {fold_idx} {split_name}'):
            try:
                img_gray = load_gray(all_paths[i])
                feat     = extract_lbp(img_gray)
                X.append(feat)
                y.append(all_labels[i])
                paths.append(all_paths[i])
            except Exception as e:
                print(f"[ERROR] {all_paths[i]}: {e}")

        X     = np.array(X,     dtype='float32')
        y     = np.array(y,     dtype=str)
        paths = np.array(paths, dtype=str)

        out_path = os.path.join(fold_dir, f'{split_name}.npz')
        np.savez_compressed(out_path, X=X, y=y, paths=paths)
        print(f"  Tersimpan: {out_path}  X.shape={X.shape}")

print("\nEkstraksi fitur selesai!")

In [ ]:
# CELL 5 - PCA (90, 95, 99%)

FEAT_SOURCES = ['lbp', 'hog', 'hog-lbp']

for feat_name in FEAT_SOURCES:
    src_dir = f'{BASE_FEATURE}/{feat_name}'
    print(f"\n{'='*50}")
    print(f"  PCA dari: {feat_name.upper()}")
    print(f"{'='*50}")

    for var, label in zip(PCA_VARS, PCA_LABELS):
        pca_dir_name = f'{feat_name}_{label}'

        for fold_idx in range(1, N_FOLDS + 1):
            out_train = f'{DIR_PCA}/{pca_dir_name}/fold_{fold_idx}/train.npz'
            out_val   = f'{DIR_PCA}/{pca_dir_name}/fold_{fold_idx}/val.npz'

            if os.path.exists(out_train) and os.path.exists(out_val):
                print(f"  [SKIP] {pca_dir_name}/fold_{fold_idx} sudah ada")
                continue

            os.makedirs(f'{DIR_PCA}/{pca_dir_name}/fold_{fold_idx}', exist_ok=True)

            # Load raw
            tr  = np.load(f'{src_dir}/fold_{fold_idx}/train.npz', allow_pickle=True)
            val = np.load(f'{src_dir}/fold_{fold_idx}/val.npz',   allow_pickle=True)

            X_tr, y_tr, p_tr   = tr['X'],  tr['y'],  tr['paths']
            X_val, y_val, p_val = val['X'], val['y'], val['paths']

            # StandardScaler
            scaler    = StandardScaler()
            X_tr_s    = scaler.fit_transform(X_tr)
            X_val_s   = scaler.transform(X_val)

            # PCA fit dari train
            pca       = PCA(n_components=var, random_state=RANDOM_STATE)
            X_tr_pca  = pca.fit_transform(X_tr_s)
            X_val_pca = pca.transform(X_val_s)

            n_comp = pca.n_components_
            print(f"  {pca_dir_name}/fold_{fold_idx} "
                  f"→ {X_tr.shape[1]}d → {n_comp}d  "
                  f"(var={var*100:.0f}%)")

            np.savez_compressed(out_train, X=X_tr_pca.astype('float32'),
                                y=y_tr, paths=p_tr)
            np.savez_compressed(out_val,   X=X_val_pca.astype('float32'),
                                y=y_val, paths=p_val)

print("\nEkstraksi PCA selesai!")

In [ ]:
# CELL 6 - Helper training: load, scale, metrics, training

# Parameter default per kernel
DEFAULT_PARAMS = {
    'linear': {'C': 10, 'class_weight': 'balanced'},
    'rbf':    {'C': 10, 'gamma': 'scale','class_weight': 'balanced'},
    'poly':   {'C': 10, 'degree': 3, 'gamma': 'scale', 'class_weight': 'balanced'},
}


def load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d['X'], d['y']


def scale_fold_data(X_tr, X_val):
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_val)


def make_svc(kernel):
    params = DEFAULT_PARAMS[kernel]
    return SVC(kernel=kernel, random_state=RANDOM_STATE, **params)


def collect_metrics(y_true, y_pred, fold, feat, scheme, kernel, params):
    rep = classification_report(y_true, y_pred,
                                 target_names=CLASSES,
                                 output_dict=True,
                                 zero_division=0)
    row = {
        'fold'           : fold,
        'feature'        : feat,
        'scheme'         : scheme,
        'kernel'         : kernel,
        'accuracy'       : round(accuracy_score(y_true, y_pred), 4),
        'precision_macro': round(precision_score(y_true, y_pred,
                                  average='macro', zero_division=0), 4),
        'recall_macro'   : round(recall_score(y_true, y_pred,
                                  average='macro', zero_division=0), 4),
        'f1_macro'       : round(f1_score(y_true, y_pred,
                                  average='macro', zero_division=0), 4),
        'best_params'    : str(params),
    }
    for cls in CLASSES:
        row[f'f1_{cls}'] = round(rep[cls]['f1-score'], 4)
    return row


In [ ]:
# CELL 7 - Training utama: HOG raw, LBP raw, HOG-LBP raw, PCA

KERNELS = ['rbf']          # sesuaikan

PIPELINES = []

for feat in ['lbp', 'hog', 'hog-lbp']:
    PIPELINES.append((feat, 'raw', f'{BASE_FEATURE}/{feat}'))

for feat in ['lbp', 'hog', 'hog-lbp']:
    for label in PCA_LABELS:
        PIPELINES.append((feat, f'pca_{label}',
                          f'{DIR_PCA}/{feat}_{label}'))

PIPELINES.insert(0, ('lbp', 'raw', DIR_LBP))

all_rows = []

for feat_name, scheme, feat_dir in PIPELINES:
    for kernel in KERNELS:
        result_folder = f'{BASE_RESULT}/{feat_name}/{kernel}/{scheme}'
        os.makedirs(result_folder, exist_ok=True)

        ckpt_path = f'{result_folder}/summary_all_folds.csv'

        if os.path.exists(ckpt_path):
            prev = pd.read_csv(ckpt_path)
            done_folds = set(prev['fold'].tolist())
            all_rows.extend(prev.to_dict('records'))
            print(f"[RESUME] {feat_name}/{kernel}/{scheme} "
                  f"— fold sudah selesai: {done_folds}")
        else:
            done_folds = set()

        print(f"\n{'='*60}")
        print(f"  {feat_name.upper()} | {scheme} | kernel={kernel}")
        print(f"{'='*60}")

        fold_rows = []

        for fold_idx in range(1, N_FOLDS + 1):
            if fold_idx in done_folds:
                continue

            X_tr,  y_tr  = load_npz(f'{feat_dir}/fold_{fold_idx}/train.npz')
            X_val, y_val = load_npz(f'{feat_dir}/fold_{fold_idx}/val.npz')

            if scheme == 'raw':
                X_tr, X_val = scale_fold_data(X_tr, X_val)

            print(f"  Fold {fold_idx}  train={X_tr.shape}  val={X_val.shape}",
                  end=' ... ')

            clf = make_svc(kernel)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_val)

            used_params = DEFAULT_PARAMS[kernel]
            row = collect_metrics(y_val, y_pred,
                                   fold_idx, feat_name, scheme, kernel,
                                   used_params)
            all_rows.append(row)
            fold_rows.append(row)

            print(f"acc={row['accuracy']:.4f}  "
                  f"f1={row['f1_macro']:.4f}  "
                  f"params={used_params}")

            pd.DataFrame(all_rows).to_csv(
                f'{BASE_RESULT}/summary_all_folds_MASTER.csv', index=False)
            pd.DataFrame(
                [r for r in all_rows
                 if r['feature']==feat_name
                 and r['scheme']==scheme
                 and r['kernel']==kernel]
            ).to_csv(ckpt_path, index=False)

        print(f"  Selesai: {feat_name}/{kernel}/{scheme}")

print("\n\nSemua training selesai!")